In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_pth=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)

from utils.transformations import reusable


### **DimUser**

### **AUTOLOADER**


In [0]:
df_user=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimUser/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storagemusicplayerde.dfs.core.windows.net/DimUser")



In [0]:
df_user=df_user.withColumn("user_name",upper(col("user_name")))


In [0]:

df_user_obj=reusable()

df_user=df_user_obj.dropColumns(df_user,["_rescued_data"])
df_user=df_user.dropDuplicates(['user_id'])




In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser")


In [0]:
df = spark.read.format("delta") \
    .load("abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimUser/data")

display(df)

user_id,user_name,country,subscription_type,start_date,end_date,updated_at
69,GREGORY HART,Syrian Arab Republic,Premium,2025-01-10,null,2025-10-05T19:49:55.000Z
129,STEPHANIE ROBINSON,Malaysia,Family,2025-04-22,null,2025-10-06T19:49:55.000Z
166,DANIEL RODRIGUEZ,Maldives,Family,2023-10-27,null,2025-09-16T19:49:55.000Z
448,DEANNA MCKINNEY,Vanuatu,Family,2024-05-13,null,2025-09-20T19:49:55.000Z
467,EMILY FRANKLIN,British Indian Ocean Territory (Chagos Archipelago),Premium,2025-05-25,null,2025-09-09T19:49:55.000Z
83,SARAH HINTON,Christmas Island,Free,2023-11-21,null,2025-09-18T19:49:55.000Z
110,RICHARD MARTINEZ,Isle of Man,Free,2023-12-19,null,2025-10-01T19:49:55.000Z
150,ABIGAIL OLSEN,Uruguay,Free,2024-03-04,null,2025-10-07T19:49:55.000Z
369,CASEY CARPENTER,Heard Island and McDonald Islands,Family,2024-07-05,null,2025-10-05T19:49:55.000Z
413,JENNIFER PAYNE,Antarctica (the territory South of 60 deg S),Premium,2025-05-03,null,2025-09-13T19:49:55.000Z


### **DimArtist**

In [0]:
df_art=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimArtist/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storagemusicplayerde.dfs.core.windows.net/DimArtist")



In [0]:
df_art_obj=reusable()

df_art=df_art_obj.dropColumns(df_art,["_rescued_data"])
df_art=df_art.dropDuplicates(['artist_id'])


In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.DimArtist")

### **DimTrack**

In [0]:
df_track= spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimTrack/checkpoint")\
        .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storagemusicplayerde.dfs.core.windows.net/DimTrack")

In [0]:
df_track=df_track.withColumn("durationFlag",when(col('duration_sec')<150,'low')\
                                            .when(col('duration_sec')<300,'medium')\
                                            .otherwise('high'))

In [0]:
df_track=df_track.withColumn('track_name',regexp_replace(col('track_name'),'-',' '))

In [0]:
df_track_obj=reusable().dropColumns(df_track,['_rescued_data'])


In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.DimTrack")


###  **DimDate**

In [0]:
df_date= spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimDate/checkpoint")\
        .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storagemusicplayerde.dfs.core.windows.net/DimDate")

In [0]:
df_date=reusable().dropColumns(df_date,['_rescued_data'])

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storagemusicplayerde.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.DimDate")

### **FactStream**



In [0]:
df_fact= spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@storagemusicplayerde.dfs.core.windows.net/FactStream/checkpoint")\
        .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storagemusicplayerde.dfs.core.windows.net/FactStream")

df_fact=reusable().dropColumns(df_fact,['_rescued_data'])

df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storagemusicplayerde.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storagemusicplayerde.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream")  